In [30]:
import numpy as np
import pandas as pd

from DataSplit import KFoldEvaluation
from ExtremeLearningMachine import ExtremeLearningMachine
from EvaluationMatrix import EvaluationMatrix


In [31]:
filePath = '../../Dataset/UCI_Gallstone_Dataset.csv'
df = pd.read_csv(filePath)

targetCol = ['Gallstone Status']
X = df.drop(targetCol, axis=1)
y = df[targetCol]

In [32]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [33]:
def cross_validation(X,y,activation_function,hidden_size=None,random_state=42, test_size=0.2, k_fold=5,regularization_lambda=0):
    if hidden_size is None:
        hidden_size = X.shape[1]

    splitter = KFoldEvaluation(random_state=random_state, test_size=test_size, k_fold=k_fold)
    x_test, y_test, folds = splitter.data_spiting(X, y)

    all_fold_results = []
    for i in range(k_fold):
        fold = folds[i]
        x_train, y_train = fold['X_train_fold'] , fold['y_train_fold']
        x_val, y_val     = fold['X_val_fold']   , fold['y_val_fold']

        elm = ExtremeLearningMachine(features_size=x_train.shape[1], hidden_size=hidden_size,activation_function=activation_function,regularization_lambda=regularization_lambda)
        elm.initialize_random_weights(random_seed=random_state)
        elm.fit(x_train, y_train)

        y_pred = elm.predict(x_val.values)
        metrics = EvaluationMatrix(y_val, y_pred)
        all_fold_results.append(metrics.get_all_metrics())

    return EvaluationMatrix.k_fold_metrics(all_fold_results)

In [34]:
def random_seed_cv_validation(X, y, activation_function, hidden_size, test_size=0.2, k_fold=5,regularization_lambda=0):
    results_list = []
    for i in range(40, 61):
        seed_result = cross_validation(
            X,
            y,
            activation_function,
            hidden_size=hidden_size,
            random_state=i,
            test_size=test_size,
            k_fold=k_fold,
            regularization_lambda=regularization_lambda
        )
        results_list.append(seed_result)

    all_results_df = pd.concat(results_list, ignore_index=True)

    return EvaluationMatrix.random_seed_metrics(all_results_df,activation_function,hidden_size,regularization_lambda)

In [35]:
def grid_search_elm(X, y, activation_function, hidden_size_range, test_size=0.2, k_fold=5):
    master_results = []
    total_runs = len(hidden_size_range)
    current_run = 1

    for h_size in hidden_size_range:
        print(f"[{current_run}/{total_runs}] | Hidden Nodes: {h_size}...")

        config_result = random_seed_cv_validation(
            X,
            y,
            activation_function=activation_function,
            hidden_size=h_size,
            test_size=test_size,
            k_fold=k_fold
        )

        master_results.append(config_result)
        current_run += 1

    final_grid_df = pd.concat(master_results, ignore_index=True)

    return final_grid_df

In [36]:
# hidden_node_cv = grid_search_elm(X,y,activation_function=sigmoid,hidden_size_range= range(44,54))

In [37]:
# # 1. Calculate the weighted score (50% Accuracy + 50% MCC)
# hidden_node_cv['Weighted_Score'] = (hidden_node_cv['avg_Accuracy_Seed_Mean'] * 0.5) + (hidden_node_cv['avg_MCC_Seed_Mean'] * 0.5)
#
# # 2. Sort by score and extract the top 5 Hidden_Nodes as a list
# best_hidden_nodes_list = hidden_node_cv.sort_values(by='Weighted_Score', ascending=False).head(5)['Hidden_Nodes'].tolist()

# 3. Output the final list
best_hidden_nodes_list = [48, 53, 51, 45, 50]

In [38]:
finalResults = []
for hiddenNodes in best_hidden_nodes_list:
    for lambdaValue in np.arange(0.00, 10.00, 0.25):
        result = random_seed_cv_validation(X,y,sigmoid,hiddenNodes,0.2,5,lambdaValue)
        finalResults.append(result)
final_grids_df = pd.concat(finalResults, ignore_index=True)
final_grids_df

,Hidden_Nodes,Activation,Lambda Value,avg_Accuracy_Seed_Mean,avg_Accuracy_Seed_Std,avg_Accuracy_Seed_Min,avg_Accuracy_Seed_Max,avg_Precision_Seed_Mean,avg_Precision_Seed_Std,avg_Precision_Seed_Min,...,avg_F1-Score_Seed_Min,avg_F1-Score_Seed_Max,avg_Balanced Accuracy_Seed_Mean,avg_Balanced Accuracy_Seed_Std,avg_Balanced Accuracy_Seed_Min,avg_Balanced Accuracy_Seed_Max,avg_MCC_Seed_Mean,avg_MCC_Seed_Std,avg_MCC_Seed_Min,avg_MCC_Seed_Max
0,48,sigmoid,0.00,0.6880,0.0222,0.6588,0.7294,0.7041,0.0260,0.6618,...,0.6327,0.7134,0.6876,0.0220,0.6578,0.7285,0.3803,0.0439,0.3185,0.4608
1,48,sigmoid,0.25,0.6891,0.0251,0.6549,0.7333,0.7073,0.0292,0.6562,...,0.6327,0.7196,0.6887,0.0249,0.6540,0.7326,0.3831,0.0497,0.3108,0.4673
2,48,sigmoid,0.50,0.6906,0.0262,0.6549,0.7373,0.7090,0.0316,0.6562,...,0.6327,0.7228,0.6902,0.0259,0.6551,0.7365,0.3862,0.0522,0.3126,0.4764
3,48,sigmoid,0.75,0.6917,0.0260,0.6549,0.7373,0.7103,0.0320,0.6614,...,0.6308,0.7207,0.6913,0.0258,0.6548,0.7362,0.3885,0.0522,0.3112,0.4760
4,48,sigmoid,1.00,0.6924,0.0255,0.6549,0.7333,0.7115,0.0315,0.6614,...,0.6381,0.7199,0.6920,0.0252,0.6548,0.7326,0.3901,0.0511,0.3112,0.4676
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,50,sigmoid,8.75,0.7109,0.0321,0.6471,0.7725,0.7332,0.0379,0.6573,...,0.6025,0.7505,0.7106,0.0322,0.6460,0.7722,0.4268,0.0651,0.3006,0.5532
196,50,sigmoid,9.00,0.7113,0.0316,0.6471,0.7725,0.7338,0.0374,0.6573,...,0.6025,0.7505,0.7109,0.0317,0.6460,0.7722,0.4276,0.0641,0.3006,0.5532
197,50,sigmoid,9.25,0.7109,0.0317,0.6471,0.7725,0.7335,0.0376,0.6573,...,0.6025,0.7505,0.7106,0.0318,0.6460,0.7722,0.4269,0.0643,0.3006,0.5532
198,50,sigmoid,9.50,0.7113,0.0320,0.6431,0.7725,0.7341,0.0378,0.6628,...,0.6002,0.7505,0.7109,0.0321,0.6422,0.7722,0.4276,0.0648,0.2928,0.5532
